In [ ]:
import pandas as pd
import polars as pl
import os

In [ ]:
INPUT_DIR = 'outputCompcor'

In [ ]:
all_temp_dfs = []
for combination in os.listdir(f'./{INPUT_DIR}/ksc'):
    if os.path.isdir(f'./{INPUT_DIR}/ksc/{combination}'):
        combination_splits = combination.split('_')
        dataset1, dataset2, repetitions = combination_splits[0], combination_splits[1], combination_splits[3]
        temp_df = pd.read_csv(f'./{INPUT_DIR}/ksc/{combination}/{combination}_ksc_metrics_measures.csv')
        temp_df = temp_df.groupby('metric').mean()
        temp_df['combination'] = f"{dataset1}_{dataset2}"
        temp_df['dataset1'] = dataset1
        temp_df['dataset2'] = dataset2
        temp_df['repetitions'] = repetitions
        all_temp_dfs.append(temp_df)

df = pd.concat(all_temp_dfs)


In [ ]:
print(df.drop(columns='Time').groupby('metric').mean(numeric_only=True).mean(axis=1).sort_values(ascending=False).head(2))
print(df.drop(columns='Time').groupby('metric').median(numeric_only=True).median(axis=1).sort_values(ascending=False).head(2))

In [ ]:
datasets = [
    'clinicalDialogueSummarizations',
    'dementiaAudio',
    'medicalAbstracts',
    'syntheticCareHomeNurseNotes',
    'simSUM'
]

filtered = df[
    df['dataset1'].isin(datasets) | df['dataset2'].isin(datasets)
]

sorted_mean_df = filtered.drop(columns='Time').groupby('metric').mean(numeric_only=True)
sorted_median_df = filtered.drop(columns='Time').groupby('metric').median(numeric_only=True)
sorted_mean_df['Overall'] = filtered.drop(columns='Time').groupby('metric').mean(numeric_only=True).mean(axis=1).tolist()
sorted_median_df['Overall'] = filtered.drop(columns='Time').groupby('metric').median(numeric_only=True).median(axis=1).tolist()
sorted_mean_df['Algorithm'] = sorted_mean_df.index
sorted_median_df['Algorithm'] = sorted_median_df.index
print("----------------- Mean -----------------")
print(pl.from_pandas(sorted_mean_df.sort_values(by='Overall', ascending=False)))
print("----------------- Median -----------------")
print(pl.from_pandas(sorted_median_df.sort_values(by='Overall', ascending=False)))